# 03 · Retrieval Is the Ceiling

The bound on everything. Measure **retrieval alone** (recall@k, MRR), then measure the
whole system's answer accuracy, and show the second can never exceed the first.

The clean experiment: feed the generator the **oracle gold passage** by hand → it answers
correctly almost always; feed it the passage the **retriever actually returned** → accuracy
falls to the retriever's recall.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Retrieval measured on its own — recall@k and MRR

In [2]:
from src.datasets.corpus import chunk_corpus
from src.pipeline.chunk import make_chunk_fn
from src.pipeline.retrieve import build_retriever
from src.evaluation.retrieval_metrics import mean_recall_at_k, mrr, hit_rate

cf = make_chunk_fn(settings.chunk.strategy, size=settings.chunk.size,
                   overlap=settings.chunk.overlap, encoder=embedder)
chunks, chunk_doc_ids = chunk_corpus(corpus, cf)
retriever = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
retriever.index(chunks)

labelled = corpus.questions_by_kind("answerable") + corpus.questions_by_kind("multi")
retrieved_docs, gold_docs = [], []
for q in labelled:
    hits = retriever.retrieve(q.text, max(10, settings.retrieve.top_k))
    retrieved_docs.append([chunk_doc_ids[h.index] for h in hits])
    gold_docs.append(list(q.gold_doc_ids))

ks = [1, 2, 3, 5, 10]
recall_curve = {k: mean_recall_at_k(retrieved_docs, gold_docs, k) for k in ks}
metrics = {"recall_at_k": recall_curve,
           "mrr": mrr(retrieved_docs, gold_docs),
           "hit_rate_at_5": hit_rate(retrieved_docs, gold_docs, 5)}
metrics


{'recall_at_k': {1: 0.875, 2: 0.9375, 3: 1.0, 5: 1.0, 10: 1.0},
 'mrr': 1.0,
 'hit_rate_at_5': 1.0}

### The ceiling: answer accuracy can never beat retrieval recall

Because the corpus is **fictional**, the model cannot answer from training priors — a
passage it never retrieved is an answer it cannot give. So if we **degrade retrieval** (by
shrinking the fixed chunk size until answers are orphaned across chunks), answer accuracy
is forced down to meet recall. Across the whole sweep, accuracy stays **on or below** the
`accuracy = recall` line — that line is the ceiling.


In [3]:
from src.datasets.corpus import chunk_corpus as _cc
from src.pipeline.rag import RagPipeline
from src.evaluation.answer_metrics import answer_correct, evidence_present


def recall_and_accuracy(chunk_fn):
    ch, _ids = _cc(corpus, chunk_fn)
    retr = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
    pipe = RagPipeline(retriever=retr, generator=generator, settings=settings, abstain=False)
    pipe.index(ch)
    qs = corpus.questions_by_kind("answerable")
    hits = rec = 0.0
    for q in qs:
        r = pipe.answer(q.text)
        hits += answer_correct(r.answer, q.answer)
        # Evidence recall = was the answer text actually retrieved (not just the document)?
        rec += evidence_present(q.answer, r.contexts)
    n = len(qs)
    return rec / n, hits / n


sweep = []
for size in [20, 40, 80, 160, 400]:
    rec, acc = recall_and_accuracy(make_chunk_fn("fixed", size=size, overlap=0))
    sweep.append({"chunk_size": size, "evidence_recall": rec, "answer_accuracy": acc})
ceiling_sweep = pd.DataFrame(sweep)
# The cap: you cannot answer correctly more often than you retrieved the evidence for.
assert (ceiling_sweep["answer_accuracy"] <= ceiling_sweep["evidence_recall"] + 1e-9).all()
ceiling_sweep


,chunk_size,evidence_recall,answer_accuracy
0,20,0.333333,0.166667
1,40,0.833333,0.666667
2,80,0.833333,0.666667
3,160,1.000000,1.000000
4,400,1.000000,1.000000


### Oracle passage vs retrieved passage

Feeding the generator the gold passage by hand (the *oracle*) versus what the retriever
returned. At full recall the two are close; note that with a tiny CPU model the oracle is
not a perfect 100% (it fumbles a fictional name even with the passage in hand), and the
extra passages in the retrieved context can even help it phrase an answer. The structural
point is unchanged: neither beats recall. The deterministic form of the invariant
(oracle ≥ retrieved with a recorded generator) is asserted in `tests/test_ceiling.py`.


In [4]:
from src.evaluation.ceiling import evaluate_ceiling

rag = RagPipeline(retriever=retriever, generator=generator, settings=settings, abstain=True)
rag.index(chunks)
ceiling = evaluate_ceiling(rag, corpus, chunk_doc_ids)
print("oracle   :", round(ceiling.oracle_accuracy, 3))
print("retrieved:", round(ceiling.retrieved_accuracy, 3))
print("recall   :", round(ceiling.retrieval_recall, 3), "(the cap)")


oracle   : 1.0
retrieved: 0.833
recall   : 1.0 (the cap)


### Figures → PNG, metrics → JSON

In [5]:
import matplotlib.pyplot as plt
from src.visualisation import ceiling_chart, grouped_lines

# The money figure: every point on or under the accuracy = evidence-recall ceiling line.
fig1, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "--", color="#333", label="accuracy = recall (ceiling)")
ax.scatter(ceiling_sweep["evidence_recall"], ceiling_sweep["answer_accuracy"],
           s=90, color="#C44E52", zorder=3)
for _, r in ceiling_sweep.iterrows():
    ax.annotate(f"{int(r['chunk_size'])}c", (r["evidence_recall"], r["answer_accuracy"]),
                textcoords="offset points", xytext=(6, -4), fontsize=8)
ax.set_xlim(-0.03, 1.05); ax.set_ylim(-0.03, 1.05)
ax.set_xlabel("evidence recall (answer text actually retrieved)")
ax.set_ylabel("answer accuracy")
ax.set_title("Retrieval is the ceiling"); ax.legend(loc="upper left")
fig1.tight_layout()

fig2 = grouped_lines(ks, {"recall@k": [recall_curve[k] for k in ks]},
                     title="Recall@k — retrieval measured alone", xlabel="k", ylabel="recall")
p1 = support.save_fig(fig1, "03_ceiling")
p2 = support.save_fig(fig2, "03_recall_at_k")
fig3 = ceiling_chart(ceiling.oracle_accuracy, ceiling.retrieved_accuracy, ceiling.retrieval_recall)
p3 = support.save_fig(fig3, "03_oracle_vs_retrieved")
support.save_json({"mode": mode, "retrieval_metrics": metrics,
                   "ceiling_sweep": ceiling_sweep.to_dict(orient="records"),
                   "ceiling": {"oracle_accuracy": ceiling.oracle_accuracy,
                               "retrieved_accuracy": ceiling.retrieved_accuracy,
                               "retrieval_recall": ceiling.retrieval_recall,
                               "n_questions": ceiling.n_questions}}, "03_ceiling")
print("saved", p1, p2, p3)
fig1


saved outputs/figures/03_ceiling.png outputs/figures/03_recall_at_k.png outputs/figures/03_oracle_vs_retrieved.png


<Figure size 550x500 with 1 Axes>